In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pl
import sns

from sklearn import metrics
from sklearn import tree
from sklearn.datasets import load_iris, load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [11]:
import pandas as pd
df = pd.read_csv("car.data")

df.head()

,vhigh,vhigh.1,2,2.1,small,low,unacc
0,vhigh,vhigh,2,2,small,med,unacc
1,vhigh,vhigh,2,2,small,high,unacc
2,vhigh,vhigh,2,2,med,low,unacc
3,vhigh,vhigh,2,2,med,med,unacc
4,vhigh,vhigh,2,2,med,high,unacc


In [12]:
columns = [
    "buying",
    "maint",
    "doors",
    "persons",
    "lug_boot",
    "safety",
    "target"
]

df = pd.read_csv("car.data", names=columns)

df.head()

,buying,maint,doors,persons,lug_boot,safety,target
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [22]:
df.isnull().sum()

buying      0
maint       0
doors       0
persons     0
lug_boot    0
safety      0
target      0
dtype: int64

In [13]:
print(df["buying"].unique())

<StringArray>
['vhigh', 'high', 'med', 'low']
Length: 4, dtype: str


In [14]:
X = df.drop("target", axis=1)
y = df[["target"]]

In [15]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(categories=[
    ["low", "med", "high", "vhigh"],
    ["low", "med", "high", "vhigh"],
    ["2", "3", "4", "5more"],
    ["2", "4", "more"],
    ["small", "med", "big"],
    ["low", "med", "high"]
])

X = encoder.fit_transform(X)

In [16]:
class_encoder = OrdinalEncoder(categories=[
    ["unacc", "acc", "good", "vgood"]
])

y = class_encoder.fit_transform(y)

In [20]:
X = X.astype(int)
y= y.astype(int)

In [23]:
y = y.ravel()

In [24]:
print(X[:5])
print(y[:5])

[[3 3 0 0 0 0]
 [3 3 0 0 0 1]
 [3 3 0 0 0 2]
 [3 3 0 0 1 0]
 [3 3 0 0 1 1]]
[0 0 0 0 0]


In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                 test_size = 0.2,
                                                 random_state = 42)

In [ ]:
#DECISION TREE

In [36]:
dtree = DecisionTreeClassifier(criterion='gini', random_state = 42)
dtree.fit(X_train, y_train)
pred = dtree.predict(X_test)
print('criterion=gini accuracy:', accuracy_score(y_test, pred))

dtree = DecisionTreeClassifier(criterion='entropy', random_state = 42)
dtree.fit(X_train, y_train)
pred = dtree.predict(X_test)
print('criterion=entropy accuracy:', accuracy_score(y_test, pred))

criterion=gini accuracy: 0.9653179190751445
criterion=entropy accuracy: 0.9653179190751445


In [37]:
print(dtree.get_depth())

12


In [38]:
max_depth = []
acc_gini = []
acc_entropy = []

# y_test_np=y_test.to_numpy()
for i in range(1,20):
    dtree = DecisionTreeClassifier(criterion='gini', max_depth=i, random_state = 42)
    dtree.fit(X_train, y_train)
    pred = dtree.predict(X_test)
    acc_gini.append(accuracy_score(y_test, pred))

    dtree = DecisionTreeClassifier(criterion='entropy', max_depth=i, random_state = 42)
    dtree.fit(X_train, y_train)
    pred = dtree.predict(X_test)
    acc_entropy.append(accuracy_score(y_test, pred))

    max_depth.append(i)

df = pd.DataFrame([acc_gini, acc_entropy, max_depth]).transpose()
df.columns=['acc_gini', 'acc_entropy', 'max_depth']
df

,acc_gini,acc_entropy,max_depth
0,0.679191,0.679191,1.0
1,0.797688,0.797688,2.0
2,0.812139,0.812139,3.0
3,0.841040,0.841040,4.0
4,0.890173,0.878613,5.0
5,0.939306,0.913295,6.0
6,0.921965,0.910405,7.0
7,0.947977,0.933526,8.0
8,0.950867,0.953757,9.0
9,0.965318,0.965318,10.0


In [58]:
#BEST
from sklearn.metrics import precision_score, recall_score, f1_score

dtree_best = DecisionTreeClassifier(max_depth = 10, criterion='gini', random_state = 42)
dtree_best.fit(X_train, y_train)
pred = dtree.predict(X_test)
print(accuracy_score(y_test, pred))
print()
print(precision_score(y_test, pred, average='macro'))
print(recall_score(y_test, pred, average='macro'))
print(f1_score(y_test, pred, average='macro'))

0.9653179190751445

0.8802856777540322
0.9090586946717352
0.8815359226824533


In [40]:
text_representation = tree.export_text(dtree_best)
print(text_representation)

|--- feature_5 <= 0.50
|   |--- class: 0
|--- feature_5 >  0.50
|   |--- feature_3 <= 0.50
|   |   |--- class: 0
|   |--- feature_3 >  0.50
|   |   |--- feature_0 <= 1.50
|   |   |   |--- feature_1 <= 1.50
|   |   |   |   |--- feature_5 <= 1.50
|   |   |   |   |   |--- feature_4 <= 0.50
|   |   |   |   |   |   |--- feature_2 <= 0.50
|   |   |   |   |   |   |   |--- feature_3 <= 1.50
|   |   |   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |   |   |--- feature_3 >  1.50
|   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |--- feature_2 >  0.50
|   |   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |--- feature_4 >  0.50
|   |   |   |   |   |   |--- feature_1 <= 0.50
|   |   |   |   |   |   |   |--- feature_2 <= 0.50
|   |   |   |   |   |   |   |   |--- feature_4 <= 1.50
|   |   |   |   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |   |   |   |--- feature_4 >  1.50
|   |   |   |   |   |   |   |   |   |--- class: 2
|   |   |   |   |   |   |   |--

In [ ]:
############################

In [ ]:
#RANDOM FOREST

In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

results = []

for n in [10, 50, 100, 200]:
    for mf in ['sqrt', 'log2', None]:

        rf = RandomForestClassifier(
            n_estimators=n,
            max_features=mf,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        pred = rf.predict(X_test)

        acc = accuracy_score(y_test, pred)

        results.append([n, mf, acc])

df = pd.DataFrame(results, columns=[
    'n_estimators',
    'max_features',
    'accuracy'
])

print(df)

    n_estimators max_features  accuracy
0             10         sqrt  0.953757
1             10         log2  0.953757
2             10          NaN  0.971098
3             50         sqrt  0.962428
4             50         log2  0.962428
5             50          NaN  0.968208
6            100         sqrt  0.962428
7            100         log2  0.962428
8            100          NaN  0.968208
9            200         sqrt  0.956647
10           200         log2  0.956647
11           200          NaN  0.968208


In [45]:
best = df.loc[df['accuracy'].idxmax()]
print(best)

n_estimators          10
max_features         NaN
accuracy        0.971098
Name: 2, dtype: object


In [57]:
from sklearn.metrics import precision_score, recall_score, f1_score

#BEST
rf = RandomForestClassifier(n_estimators=10, max_features=None, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print(accuracy_score(y_test, pred))
print()
print(precision_score(y_test, pred, average='macro'))
print(recall_score(y_test, pred, average='macro'))
print(f1_score(y_test, pred, average='macro'))

0.9710982658959537

0.883679240057605
0.958185683912119
0.9085957196880927


In [50]:
###############################################################

In [51]:
#NEURAL NETWORK

In [52]:
# Scale the features (important for neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [59]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

results = []

activations = ['relu', 'tanh', 'sigmoid']
learning_rates = [0.1, 0.01, 0.001]

for act in activations:
    for lr in learning_rates:

        model = Sequential([

            Dense(32, activation=act, input_dim=X_train_scaled.shape[1]),
            Dropout(0.2),

            Dense(16, activation=act),
            Dropout(0.2),

            Dense(4, activation='softmax')
        ])

        model.compile(
            optimizer=Adam(learning_rate=lr),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        model.fit(
            X_train_scaled,
            y_train,
            epochs=20,
            batch_size=32,
            verbose=0
        )

        pred = model.predict(X_test_scaled, verbose=0)

        pred = np.argmax(pred, axis=1)

        acc = accuracy_score(y_test, pred)

        results.append([act, lr, acc])

df = pd.DataFrame(results, columns=[
    'activation',
    'learning_rate',
    'accuracy'
])

print(df)

best = df.loc[df['accuracy'].idxmax()]

print("\nBEST CONFIGURATION:")
print(best)

C:\Users\Sandra\Downloads\Flight_Management_DNICK_2026-main\Flight_Management_DNICK_2026-main\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Sandra\Downloads\Flight_Management_DNICK_2026-main\Flight_Management_DNICK_2026-main\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\Sandra\Downloads\Flight_Management_DNICK_2026-main\Flight_Management_DNICK_2026-main\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `

  activation  learning_rate  accuracy
0       relu          0.100  0.924855
1       relu          0.010  0.971098
2       relu          0.001  0.913295
3       tanh          0.100  0.872832
4       tanh          0.010  0.933526
5       tanh          0.001  0.835260
6    sigmoid          0.100  0.962428
7    sigmoid          0.010  0.907514
8    sigmoid          0.001  0.777457

BEST CONFIGURATION:
activation           relu
learning_rate        0.01
accuracy         0.971098
Name: 1, dtype: object


In [60]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

def build_model(input_dim, activation='relu', lr=0.01):

    model = Sequential([
        Input(shape=(input_dim,)),

        Dense(32, activation=activation),
        Dropout(0.2),

        Dense(16, activation=activation),
        Dropout(0.2),

        Dense(4, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [61]:
model = build_model(X_train_scaled.shape[1], activation='relu', lr=0.01)

model.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=0
)

pred = model.predict(X_test_scaled, verbose=0)
pred = pred.argmax(axis=1)

In [62]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(accuracy_score(y_test, pred))
print()
print(precision_score(y_test, pred, average='macro'))
print(recall_score(y_test, pred, average='macro'))
print(f1_score(y_test, pred, average='macro'))

0.9682080924855492

0.9477835442565733
0.9454280199647149
0.9437149859943977
